# 04 — Embedding congelati a parità di genere (Versione 3)

Gli embedding di e5-large e bge-large, uno per articolo, in `../Results/2-classificazione/embedding/`: se ci sono
(calcolati nella Versione 2, sugli stessi articoli) si riusano, altrimenti si calcolano qui (primi 512 token,
congelati, come nella Versione 2). Ogni testata è la media dei vettori dei suoi articoli; sopra, una
regressione logistica con i **pesi a parità di genere** (testate pesate, un gruppo vale al più 5) e C scelto in una
validazione interna per gruppi, sul solo training, con la metrica principale. Test sulle testate valutate; ogni
articolo riceve la probabilità della sua testata. Aggiunto la sera del 20 ottobre.

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler

from comune import RESULTS, SEED, carica, fattori_genere, metriche, per_testata, pesi_righe, registratore

In [ ]:
NOME = "e5_genere"
EMB = "e5_large_v2"
C_GRIGLIA = "0.01,0.1,1,10"
PROVA = False

In [ ]:
OUT = (RESULTS / "prova" if PROVA else RESULTS) / NOME
OUT.mkdir(parents=True, exist_ok=True)
registra = registratore(OUT.parent / f"avanzamento_{NOME}.log")
art = carica(PROVA)
D = Path("../Results/2-classificazione/embedding"); D.mkdir(parents=True, exist_ok=True)
if not (D / f"{EMB}.npy").exists():          # come nella Versione 2: 512 token, e5 con "query: " e media, bge con CLS
    import torch
    from transformers import AutoModel, AutoTokenizer
    MODELLO_EMB = {"e5_large_v2": "intfloat/e5-large-v2", "bge_large_v15": "BAAI/bge-large-en-v1.5"}[EMB]
    prefisso, cls = ("query: ", False) if EMB.startswith("e5") else ("", True)
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tk = AutoTokenizer.from_pretrained(MODELLO_EMB); md = AutoModel.from_pretrained(MODELLO_EMB).to(dev).eval()
    md = md.half() if dev.type == "cuda" else md
    testi = [prefisso + x for x in art.text]; ordine = np.argsort([len(x) for x in testi])
    E = np.zeros((len(testi), md.config.hidden_size), dtype=np.float32)
    with torch.no_grad():
        for s0 in range(0, len(testi), 64):
            idx = ordine[s0:s0 + 64]
            enc = tk([testi[i] for i in idx], truncation=True, max_length=512, padding=True, return_tensors="pt").to(dev)
            h = md(**enc).last_hidden_state.float()
            v = h[:, 0] if cls else (h * enc.attention_mask.unsqueeze(-1)).sum(1) / enc.attention_mask.sum(1, keepdim=True)
            E[idx] = torch.nn.functional.normalize(v, dim=1).cpu().numpy()
    np.save(D / f"{EMB}.npy", E); np.save(D / f"{EMB}_article_id.npy", art.article_id.to_numpy())
    registra(f"embedding {EMB} calcolati")
E, ids = np.load(D / f"{EMB}.npy"), np.load(D / f"{EMB}_article_id.npy", allow_pickle=True)
riga = pd.Series(np.arange(len(ids)), index=ids)
art = art[art.article_id.isin(riga.index)].reset_index(drop=True)
Ea = E[riga.loc[art.article_id].to_numpy()]
t = art.groupby("domain").agg(y=("y", "first"), gruppo=("gruppo", "first"), fold=("fold_gruppo", "first"), peso=("peso", "first"),
                              orientamento=("orientamento", "first"), valutazione=("valutazione", "first")).reset_index()
pos = art.groupby("domain").indices
X = np.stack([Ea[pos[d]].mean(0) for d in t.domain]); X /= np.linalg.norm(X, axis=1, keepdims=True)
registra(f"=== {NOME}: {EMB}, {len(art)} articoli, {len(t)} testate ({t.valutazione.sum()} valutate) ===")


def adatta(Xa, ta, c):
    sc = StandardScaler().fit(Xa)
    w = pesi_righe(ta, fattori_genere(ta))
    return sc, LogisticRegression(C=c, max_iter=5000, random_state=SEED).fit(sc.transform(Xa), ta.y, sample_weight=w)


griglia = [float(c) for c in C_GRIGLIA.split(",")]
p = np.full(len(t), np.nan)
for f in sorted(t.fold.unique()):
    tr, te = np.where(t.fold != f)[0], np.where((t.fold == f) & t.valutazione)[0]
    ttr = t.iloc[tr].reset_index(drop=True); punti = {c: [] for c in griglia}
    for itr, iva in StratifiedGroupKFold(3, shuffle=True, random_state=SEED).split(ttr, ttr.orientamento + ttr.y.astype(str), ttr.gruppo):
        iva = iva[ttr.valutazione.to_numpy()[iva]]
        for c in griglia:
            sc, clf = adatta(X[tr][itr], ttr.iloc[itr], c)
            v = ttr.iloc[iva].assign(p=clf.predict_proba(sc.transform(X[tr][iva]))[:, 1])
            punti[c].append(metriche(v.set_index("domain"))["media sugli orientamenti"])
    medie = {c: float(np.nanmean(s)) for c, s in punti.items()}; C = max(medie, key=medie.get)
    sc, clf = adatta(X[tr], t.iloc[tr], C)
    p[te] = clf.predict_proba(sc.transform(X[te]))[:, 1]
    m = metriche(t.iloc[te].assign(p=p[te]).set_index("domain"))
    registra(f"fold {f}: C {C} (interne {', '.join(f'{k}: {x:.3f}' for k, x in medie.items())}); media sugli orientamenti {m['media sugli orientamenti']:.3f}")
pt = pd.Series(p, index=t.domain)
av = art[art.valutazione]
pred = pd.DataFrame({"article_id": av.article_id, "domain": av.domain, "fold": av.fold_gruppo, "y": av.y,
                     "orientamento": av.orientamento, "p": av.domain.map(pt).to_numpy()})
pred.to_csv(OUT / "articoli.csv", index=False)
m = metriche(per_testata(pred)); json.dump(m, open(OUT / "metriche.json", "w"), indent=1)
(OUT / "FATTO").write_text(time.strftime("%Y-%m-%d %H:%M:%S"))
registra(f"=== {NOME} FINITO: media sugli orientamenti {m['media sugli orientamenti']:.3f}, AUC {m['AUC media sugli orientamenti']:.3f} ===")
pd.Series(m).round(3)